# RMDS Hackathon — Dataset A: Seasonal vaccine, cost-minimising targeting

**Goal:** decide *who gets a vaccine sent home* so the town's total spend (vaccines + compensation + H3N2 hospitalisations) is as low as possible.

**Cost per person** (V = \$300 vaccine, H = \$18,500 hospitalisation, everyone gets infected):

| | Accepts vaccine (y=1) | Refuses (y=0) |
|---|---|---|
| **Send** | V + h_vacc·H | V (wasted) + h_unvacc·H |
| **Don't send** | V (compensation) + h_unvacc·H | h_unvacc·H |

**Optimal rule:** send if `P(accept) > V / (V + (h_unvacc − h_vacc)·H)` — a **different threshold per person**, low for high-risk people, high for low-risk people.

**How to use:** set `GROUP` and the file paths in the next cell → *Runtime ▸ Run all*. The 5 submission CSVs are written to the working folder (and auto-downloaded in Colab).

In [ ]:
# ===================== CONFIG — edit me =====================
GROUP = 0                                   # <-- your group number
TRAIN_PATH  = "dataset_A_train.csv"         # training features (+ target if in the same file)
LABELS_PATH = None                          # e.g. "dataset_A_train_labels.csv" if labels are separate
TEST_PATH   = "dataset_A_test.csv"          # test features
EXAMPLE_SUB_PATH = "dataset_A_example_submission.csv"  # optional: used to copy exact column names

ID_COL = "respondent_id"
TARGET = "seasonal_vaccine"
VACCINE_COST = 300
HOSP_COST = 18_500
SEED = 42
N_FOLDS = 5
# ============================================================

In [ ]:
import os, glob
needed = [p for p in [TRAIN_PATH, LABELS_PATH, TEST_PATH] if p]
missing = [p for p in needed if not os.path.exists(p)]
if missing:
    try:
        from google.colab import files  # Colab: upload the CSVs
        print("Please upload:", missing)
        files.upload()
    except ImportError:
        pass
print("CSV files here:", glob.glob("*.csv"))
missing = [p for p in needed if not os.path.exists(p)]
assert not missing, f"Still missing {missing} — fix the paths in the CONFIG cell."

## 1. Load & explore

In [ ]:
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

train = pd.read_csv(TRAIN_PATH)
if LABELS_PATH:
    labels = pd.read_csv(LABELS_PATH)
    train = train.merge(labels[[ID_COL, TARGET]], on=ID_COL, how="inner")
test = pd.read_csv(TEST_PATH)

assert TARGET in train.columns, f"'{TARGET}' not found — set LABELS_PATH?"
LEAKY = {TARGET, "h1n1_vaccine", ID_COL}
feature_cols = [c for c in train.columns if c not in LEAKY and c in test.columns]
X, X_test = train[feature_cols].copy(), test[feature_cols].copy()
y = train[TARGET].astype(int).values

cat_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"train {train.shape}, test {test.shape}")
print(f"{len(num_cols)} numeric, {len(cat_cols)} categorical features")
print(f"acceptance rate: {y.mean():.3f}")
display(X.isna().mean().sort_values(ascending=False).head(12).rename("missing %").to_frame().style.format("{:.1%}"))

In [ ]:
def rate_by(col, ax):
    r = train.groupby(train[col].fillna("missing").astype(str))[TARGET].agg(["mean", "size"])
    ax.barh(r.index, r["mean"], color="#3b6ea5")
    ax.set_title(f"Acceptance by {col}", fontsize=10); ax.set_xlim(0, 1)
    for i, (m, n) in enumerate(zip(r["mean"], r["size"])):
        ax.text(m + 0.01, i, f"{m:.0%} (n={n})", va="center", fontsize=8)

plot_cols = [c for c in ["age_group", "doctor_recc_seasonal", "opinion_seas_vacc_effective",
                         "opinion_seas_risk", "number_chronic_med_conditions", "health_insurance"]
             if c in train.columns]
fig, axes = plt.subplots(int(np.ceil(len(plot_cols) / 2)), 2, figsize=(13, 3.2 * np.ceil(len(plot_cols) / 2)))
for c, ax in zip(plot_cols, axes.ravel()):
    rate_by(c, ax)
for ax in axes.ravel()[len(plot_cols):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Hospitalisation risk model (the "use generative AI for this" part)

These are **assumptions** — change them after cross-checking with other AIs / sources, and say where they came from in your presentation.

* **Base risk by age** (P(hospitalised | infected, unvaccinated)). Anchor: CDC 2017–18 (H3N2-dominated) ≈ 41M illnesses → 710k hospitalisations ≈ **1.7 %** overall, heavily concentrated in 65+.
* **Chronic conditions**: each condition multiplies risk by `chronic_rr`.
* **Vaccine effectiveness vs H3N2 hospitalisation**: HAIVEN study pooled 2016–18 ≈ **23 %** (H3N2 VE is usually low).
* Scenarios let you test how robust each submission is to these guesses.

In [ ]:
# Base P(hospitalisation | infected, unvaccinated) by lower bound of age band — EDIT after checking sources
AGE_BASE_RISK = {18: 0.005, 35: 0.007, 45: 0.012, 55: 0.020, 65: 0.080}

SCENARIOS = {
    "base":      dict(ve=0.23, risk_mult=1.0, chronic_rr=1.8),
    "ve_high":   dict(ve=0.40, risk_mult=1.0, chronic_rr=1.8),
    "ve_low":    dict(ve=0.10, risk_mult=1.0, chronic_rr=1.8),
    "risk_high": dict(ve=0.23, risk_mult=1.5, chronic_rr=2.2),
    "risk_low":  dict(ve=0.23, risk_mult=0.6, chronic_rr=1.5),
}

def age_lower(v):
    m = re.search(r"\d+", str(v))
    return int(m.group()) if m else None

def hosp_risk(df, ve, risk_mult, chronic_rr):
    # returns (h_unvacc, h_vacc) arrays for each row
    keys = sorted(AGE_BASE_RISK)
    def base_for(v):
        a = age_lower(v)
        if a is None:
            return np.nan
        k = max([k for k in keys if k <= a], default=keys[0])
        return AGE_BASE_RISK[k]
    if "age_group" in df.columns:
        base = df["age_group"].map(base_for).astype(float)
    else:
        base = pd.Series(np.nan, index=df.index)
    base = base.fillna(np.nanmean(list(AGE_BASE_RISK.values())))
    if "number_chronic_med_conditions" in df.columns:
        n = pd.to_numeric(df["number_chronic_med_conditions"], errors="coerce").fillna(0).clip(0, 3)
    else:
        n = 0
    h_u = np.clip(base * risk_mult * chronic_rr ** n, 0, 0.6).values
    return h_u, h_u * (1 - ve)

H_TRAIN = {s: hosp_risk(X, **kw) for s, kw in SCENARIOS.items()}
H_TEST  = {s: hosp_risk(X_test, **kw) for s, kw in SCENARIOS.items()}

h_u, h_v = H_TRAIN["base"]
thr = VACCINE_COST / (VACCINE_COST + (h_u - h_v) * HOSP_COST)
print(f"mean unvaccinated hospitalisation risk (base): {h_u.mean():.3%}")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].hist(h_u, bins=40, color="#c0504d"); ax[0].set_title("Per-person hospitalisation risk (base)")
ax[1].hist(thr, bins=40, color="#3b6ea5"); ax[1].set_title("Per-person send threshold on P(accept)")
plt.tight_layout(); plt.show()

## 3. Cost function & decision rules

In [ ]:
V, H = VACCINE_COST, HOSP_COST

def total_cost(y_true, send, h_u, h_v):
    # Total town spend for a send/don't-send decision vector
    y_true = np.asarray(y_true).astype(bool); send = np.asarray(send).astype(bool)
    vaccine_spend = V * send.sum() + V * (~send & y_true).sum()   # doses posted + compensation
    vaccinated = send & y_true
    hosp_spend = H * np.where(vaccinated, h_v, h_u).sum()
    return vaccine_spend + hosp_spend

def bayes_send(p, h_u, h_v):
    # Send if expected cost of sending < expected cost of not sending
    return p > V / (V + (h_u - h_v) * H)

def best_global_threshold(p, y_true, h_u, h_v):
    grid = np.linspace(0.05, 0.99, 95)
    costs = [total_cost(y_true, p > t, h_u, h_v) for t in grid]
    return grid[int(np.argmin(costs))]

## 4. Models — 5-fold out-of-fold probabilities

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

def make_logreg():
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num_cols),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"),
                              OneHotEncoder(handle_unknown="ignore")), cat_cols),
    ])
    return make_pipeline(pre, LogisticRegression(C=0.3, max_iter=3000))

def make_hgb():
    pre = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), cat_cols),
        ("num", "passthrough", num_cols),
    ])
    model = HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=300, max_leaf_nodes=31, min_samples_leaf=40,
        l2_regularization=1.0, categorical_features=list(range(len(cat_cols))), random_state=SEED)
    return make_pipeline(pre, model)

MODELS = {"logreg": make_logreg, "hgb": make_hgb}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof = {m: np.zeros(len(y)) for m in MODELS}
for fold, (tr, va) in enumerate(skf.split(X, y)):
    for m, make in MODELS.items():
        est = make().fit(X.iloc[tr], y[tr])
        oof[m][va] = est.predict_proba(X.iloc[va])[:, 1]
    print(f"fold {fold + 1}/{N_FOLDS} done")
oof["blend"] = 0.5 * oof["logreg"] + 0.5 * oof["hgb"]

pd.DataFrame({m: {"ROC AUC": roc_auc_score(y, p), "log loss": log_loss(y, p), "Brier": brier_score_loss(y, p)}
              for m, p in oof.items()}).T.round(4)

In [ ]:
# Calibration check — the per-person threshold assumes probabilities are honest
from sklearn.calibration import calibration_curve
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot([0, 1], [0, 1], "k--", lw=1)
for m, p in oof.items():
    fy, fx = calibration_curve(y, p, n_bins=10)
    ax.plot(fx, fy, marker="o", label=m)
ax.set_xlabel("predicted P(accept)"); ax.set_ylabel("observed rate"); ax.legend(); ax.set_title("Reliability")
plt.show()

## 5. Compare strategies on £/\$ cost

Each candidate = model × decision rule. Every candidate is costed under **every** scenario; we rank by the **mean cost across scenarios** (robust to our hospitalisation guesses).

In [ ]:
candidates = {}   # name -> dict(model, rule, param)
for m in oof:
    for s in SCENARIOS:
        candidates[f"{m} | per-person thr ({s})"] = dict(model=m, rule="bayes", scen=s)
    t = best_global_threshold(oof[m], y, *H_TRAIN["base"])
    candidates[f"{m} | global thr {t:.2f}"] = dict(model=m, rule="global", t=t)
    candidates[f"{m} | thr 0.50"] = dict(model=m, rule="global", t=0.5)

def decide(c, p, H_sets):
    if c["rule"] == "bayes":
        return bayes_send(p, *H_sets[c["scen"]])
    return p > c["t"]

rows = {}
for name, c in candidates.items():
    send = decide(c, oof[c["model"]], H_TRAIN)
    rows[name] = {s: total_cost(y, send, *H_TRAIN[s]) for s in SCENARIOS} | {"sent %": send.mean()}
# reference points
for name, send in {"BASELINE send nobody": np.zeros(len(y), bool),
                   "BASELINE send everyone": np.ones(len(y), bool),
                   "ORACLE perfect knowledge": y.astype(bool)}.items():
    rows[name] = {s: total_cost(y, send, *H_TRAIN[s]) for s in SCENARIOS} | {"sent %": send.mean()}

res = pd.DataFrame(rows).T
res["mean cost"] = res[list(SCENARIOS)].mean(axis=1)
res["per person"] = res["mean cost"] / len(y)
res = res.sort_values("mean cost")
display(res.style.format({**{s: "${:,.0f}" for s in SCENARIOS}, "mean cost": "${:,.0f}",
                          "per person": "${:,.2f}", "sent %": "{:.1%}"}))

In [ ]:
top = res.head(12).iloc[::-1]
colors = ["#9bbb59" if i.startswith("ORACLE") else "#c0504d" if i.startswith("BASELINE") else "#3b6ea5" for i in top.index]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top.index, top["per person"], color=colors)
ax.set_xlabel("mean cost per person across scenarios ($)")
ax.set_xlim(top["per person"].min() * 0.97, top["per person"].max() * 1.01)
ax.set_title("Best strategies (blue = models, green = oracle, red = baselines)")
plt.tight_layout(); plt.show()

## 6. Fit on all training data → write the 5 best submissions

In [ ]:
test_p = {}
for m, make in MODELS.items():
    test_p[m] = make().fit(X, y).predict_proba(X_test)[:, 1]
test_p["blend"] = 0.5 * test_p["logreg"] + 0.5 * test_p["hgb"]

sub_cols = [ID_COL, TARGET]
if EXAMPLE_SUB_PATH and os.path.exists(EXAMPLE_SUB_PATH):
    sub_cols = pd.read_csv(EXAMPLE_SUB_PATH, nrows=1).columns.tolist()
    print("Using example submission columns:", sub_cols)

ranked = [n for n in res.index if n in candidates][:5]
written = []
for order, name in enumerate(ranked, start=1):
    c = candidates[name]
    send = decide(c, test_p[c["model"]], H_TEST).astype(int)
    sub = pd.DataFrame({sub_cols[0]: test[ID_COL].values, sub_cols[1]: send})
    fname = f"challenge_submission_group_{GROUP}_order_{order}.csv"
    sub.to_csv(fname, index=False)
    written.append(fname)
    print(f"order {order}: {name:45s} → {fname}  (vaccines sent: {send.sum()} / {len(send)})")

try:
    from google.colab import files
    for f in written:
        files.download(f)
except ImportError:
    pass

## 7. What drives acceptance? (quick explainability — expand after the hackathon)

In [ ]:
from sklearn.inspection import permutation_importance
idx = np.random.RandomState(SEED).choice(len(X), size=min(5000, len(X)), replace=False)
final_hgb = make_hgb().fit(X, y)
pi = permutation_importance(final_hgb, X.iloc[idx], y[idx], scoring="roc_auc", n_repeats=3, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=feature_cols).sort_values().tail(15)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(imp.index, imp.values, color="#3b6ea5"); ax.set_title("Permutation importance (drop in ROC AUC)")
plt.tight_layout(); plt.show()

### Notes for the presentation
* **Why not accuracy?** A wasted dose costs \$300; missing someone costs \$300 compensation **plus** their lost protection. The per-person threshold trades these off explicitly.
* **Why the threshold is high for most people:** H3N2 vaccine effectiveness against hospitalisation is low (~23 %), so for a young healthy person the vaccine only saves ~\$20 of expected hospital cost — only send it if they're almost sure to accept.
* **Robustness:** ranking uses mean cost across 5 assumption scenarios. Tune `AGE_BASE_RISK` / `SCENARIOS` after cross-checking with other AIs and sources.
* Out-of-fold estimates for the tuned global threshold are slightly optimistic (threshold chosen on the same predictions).